In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('./').resolve().parent.parent))
import json

from SUPPORT.utils.train_collection import TrainingClass
from SUPPORT.utils.test_collection import TestingClass

In [2]:
in_path = "/data/garfinkeljb/noisy_data/CDH5_109_M_20250501_FOV2_4D_Ch2"
exp_name = None
dimensionality = "3+1D" # options: "2+1D" (planar + time), "3D" (static volumetric), or "3+1D" (volumetric + time)
rm_dir = True # if True, do not preserve checkpoints (saves local storage memory)

In [3]:
# personalization: one-time change, keep after initialization
out_dir = None
# node-dependent: change if deviating from typical node specs
n_cpu = 8
gpu = '0'

In [17]:
with open('./parameters.json', 'r') as file: params = json.load(file)
train_params = params[dimensionality]['train']
test_params = params[dimensionality]['test']

train_params['dimensionality'] = dimensionality
train_params['rm_dir'] = rm_dir
train_params['out_dir'] = out_dir
train_params['n_cpu'] = n_cpu
train_params['cuda_device'] = gpu
train_params['in_path'] = in_path
train_params['exp_name'] = exp_name

train_dict = {
    'in_path': train_params['in_path'],
    'exp_name': train_params['exp_name'],
    'out_dir': train_params['out_dir'],
    'dimensionality': train_params['dimensionality'],
    'rm_dir': train_params['rm_dir'],
    'n_epochs': train_params['n_epochs'],
    'cuda_device': train_params['cuda_device'],
    'patch_size':train_params['patch_size'],
    'patch_interval':train_params['patch_interval'],
    'batch_size': train_params['batch_size'],
    'depth': train_params['depth'],
    'bc_channels': train_params['bc_channels'],
    'obo_channels': train_params['obo_channels'],
    'll_channels': train_params['ll_channels'],
    'unet_channels': train_params['unet_channels'],
    'bs_size': train_params['bs_size'],
    'lr': train_params['lr'],
    'loss_coef': train_params['loss_coef'],
    'n_cpu': train_params['n_cpu'],
    'prefetch_factor': train_params['prefetch_factor']
}
tcd = TrainingClass(train_dict)
tcd.vars_dict()

Training parameters -----> 
in_path                        /data/garfinkeljb/noisy_data/CDH5_109_M_20250501_FOV2_4D_Ch2 
exp_name                       None 
out_dir                        None 
dimensionality                 3+1D 
rm_dir                         True 
n_epochs                       20 
cuda_device                    0 
patch_size                     [61, 128, 128] 
patch_interval                 [1, 64, 64] 
batch_size                     256 
depth                          5 
bc_channels                    64 
obo_channels                   [32, 16] 
ll_channels                    [64, 32, 16] 
unet_channels                  [64, 128, 256, 512, 1024] 
bs_size                        [3, 3] 
lr                             5e-05 
loss_coef                      [0.5, 0.5] 
n_cpu                          8 
prefetch_factor                2 


In [ ]:
tcd.run()

In [18]:
epoch = vars(tcd)['n_epochs']
model_path = vars(tcd)['pth_dir'] + f'/model{epoch:03d}.pth'

In [20]:
test_params['model_path'] = model_path
test_params['out_dir'] = vars(tcd)['out_dir']
test_params['exp_name'] = vars(tcd)['exp_name']
test_dict = {
    'in_path':          train_params['in_path'],
    'model_path':       test_params['model_path'],
    'exp_name':         test_params['exp_name'],
    'out_dir':          test_params['out_dir'],
    'dimensionality':   train_params['dimensionality'],
    'cuda_device':      train_params['cuda_device'],
    'patch_size':       train_params['patch_size'],
    'patch_interval':   test_params['patch_interval'],
    'batch_size':       train_params['batch_size'],
    'depth':            train_params['depth'],
    'bc_channels':      train_params['bc_channels'],
    'obo_channels':     train_params['obo_channels'],
    'll_channels':      train_params['ll_channels'],
    'unet_channels':    train_params['unet_channels'],
    'bs_size':          train_params['bs_size'],
    'n_cpu':            train_params['n_cpu'],
    'prefetch_factor':  train_params['prefetch_factor'],
    'pad':              test_params['pad']
}
tcd = TestingClass(test_dict)

Training parameters -----> 
in_path                        /data/garfinkeljb/noisy_data/CDH5_109_M_20250501_FOV2_4D_Ch2 
model_path                     /Users/garfinkeljb/Documents/Denoising Models/SUPPORT/notebooks/tmp/CDH5_109_M_20250501_FOV2_4D_Ch2/pth/model020.pth 
exp_name                       CDH5_109_M_20250501_FOV2_4D_Ch2 
out_dir                        /Users/garfinkeljb/Documents/Denoising Models/SUPPORT/notebooks/outputs 
dimensionality                 3+1D 
cuda_device                    0 
patch_size                     [61, 128, 128] 
patch_interval                 [1, 64, 64] 
batch_size                     256 
depth                          5 
bc_channels                    64 
obo_channels                   [32, 16] 
ll_channels                    [64, 32, 16] 
unet_channels                  [64, 128, 256, 512, 1024] 
bs_size                        [3, 3] 
n_cpu                          8 
prefetch_factor                2 
pad                            True 


In [ ]:
tcd.run()